In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Sliding windows
def make_sliding_windows(y, history, horizon, step=1):
    y = np.asarray(y)
    X, Y, origins = [], [], []
    for t in range(history, len(y) - horizon + 1, step):
        X.append(y[t-history:t])
        Y.append(y[t:t+horizon])
        origins.append(t)
    return np.array(X), np.array(Y), np.array(origins)


In [ ]:
# Time‑ordered train / test split
def split_train_test_windows(X, Y, origins, test_ratio=0.2):
    split = int(len(X) * (1 - test_ratio))
    return (
        X[:split], Y[:split],
        X[split:], Y[split:],
        origins[:split], origins[split:]
    )


In [ ]:
# Metrics (shared across ALL models)
def evaluate_forecasts(results):
    yt, yp, lo, up = [], [], [], []
    for r in results:
        yt.extend(r["y_true"])
        yp.extend(r["y_pred"])
        lo.extend(r["lower"])
        up.extend(r["upper"])
    yt, yp, lo, up = map(np.array, [yt, yp, lo, up])
    rmse = float(np.sqrt(np.mean((yt - yp) ** 2)))
    coverage = float(np.mean((yt >= lo) & (yt <= up)))
    return rmse, coverage

def evaluate_by_horizon(results):
    H = len(results[0]["y_true"])
    rmse_h, cov_h = [], []
    for h in range(H):
        yt, yp, lo, up = [], [], [], []
        for r in results:
            yt.append(r["y_true"][h])
            yp.append(r["y_pred"][h])
            lo.append(r["lower"][h])
            up.append(r["upper"][h])
        yt, yp, lo, up = map(np.array, [yt, yp, lo, up])
        rmse_h.append(np.sqrt(np.mean((yt - yp) ** 2)))
        cov_h.append(np.mean((yt >= lo) & (yt <= up)))
    return rmse_h, cov_h

def compute_winkler(results, alpha=0.1):
    scores = []
    for r in results:
        for y, l, u in zip(r["y_true"], r["lower"], r["upper"]):
            width = u - l
            if y < l:
                scores.append(width + 2/alpha * (l - y))
            elif y > u:
                scores.append(width + 2/alpha * (y - u))
            else:
                scores.append(width)
    return float(np.mean(scores))

def compute_sample_rmse(results):
    errs = []
    for i, r in enumerate(results):
        rmse = np.sqrt(np.mean((np.array(r["y_true"]) - np.array(r["y_pred"]))**2))
        errs.append((i, rmse))
    return sorted(errs, key=lambda x: x[1])


In [ ]:
# DeepAR model

class DeepAR(nn.Module):
    def __init__(self, hidden_size=1024, num_layers=2, horizon=28):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0
        )
        self.dropout = nn.Dropout(0.1)
        self.fc_mu = nn.Linear(hidden_size, horizon)
        self.fc_sigma = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        h = self.dropout(h_n[-1])
        mu = self.fc_mu(h)
        sigma = F.softplus(self.fc_sigma(h)) + 1e-6
        return mu, sigma


In [ ]:
# Gaussian NLL loss

def gaussian_nll(mu, sigma, y_true):
    return torch.mean(
        0.5 * torch.log(2 * torch.pi * sigma**2)
        + 0.5 * ((y_true - mu)**2) / (sigma**2)
    )

In [ ]:
# Plot helpers
def plot_single_forecast(results, sid, title):
    r = results[sid]
    full = np.concatenate([r["history"], r["y_true"]])
    t_all = np.arange(len(full))
    t_fut = np.arange(len(r["history"]), len(full))
    plt.figure(figsize=(10,4))
    plt.plot(t_all, full, color="black")
    plt.plot(t_fut, r["y_pred"], color="red")
    plt.fill_between(t_fut, r["lower"], r["upper"], alpha=0.3)
    plt.axvline(len(r["history"])-1, linestyle="--", color="gray")
    plt.title(title)
    plt.tight_layout()
    plt.show()

def plot_best_worst_forecasts_deepar(results, name, n_show=3):
    errs = compute_sample_rmse(results)
    for sid, e in errs[:n_show]:
        plot_single_forecast(results, sid, f"DeepAR Best (RMSE={e:.2f})")
    for sid, e in errs[-n_show:]:
        plot_single_forecast(results, sid, f"DeepAR Worst (RMSE={e:.2f})")

def plot_coverage_train_vs_test(train, test, name, target=0.9):
    h = range(1, len(test)+1)
    plt.figure(figsize=(8,5))
    plt.plot(h, train, marker="o", label="Train")
    plt.plot(h, test, marker="o", label="Test")
    plt.axhline(target, linestyle="--", color="red", label="Target")
    plt.xlabel("Forecast Horizon")
    plt.ylabel("Coverage")
    plt.title(f"DeepAR Coverage vs Horizon\n{name}")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# DeepAR pipeline (TRAIN + TEST)

def run_deepar_pipeline(
    y,
    test_ratio=0.2,
    hidden_size=128,
    epochs=400,
    lr=1e-3,
    mc_samples=50,
):
    history = min(100, len(y)//3)
    horizon = min(28, len(y)//6)
    if len(y) < history + horizon + 1:
        return None

    X, Y, orig = make_sliding_windows(y, history, horizon)
    X_tr, Y_tr, X_te, Y_te, _, _ = split_train_test_windows(X, Y, orig, test_ratio)
    if len(X_tr) == 0 or len(X_te) == 0:
        return None

    scale = X_tr.mean()

    X_trs = (X_tr - scale) / scale
    X_tes = (X_te - scale) / scale
    Y_trs = Y_tr / scale

    Xt = torch.tensor(X_trs[..., None], dtype=torch.float32)
    Yt = torch.tensor(Y_trs, dtype=torch.float32)
    Xs = torch.tensor(X_tes[..., None], dtype=torch.float32)

    model = DeepAR(hidden_size=hidden_size, horizon=horizon)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for _ in range(epochs):
        mu, sigma = model(Xt)
        loss = gaussian_nll(mu, sigma, Yt)
        opt.zero_grad()
        loss.backward()
        opt.step()

    model.train() 
    mus_tr, sig_tr = [], []
    with torch.no_grad():
        for _ in range(mc_samples):
            mu_tr, s_tr = model(Xt)
            mus_tr.append(mu_tr.numpy())
            sig_tr.append(s_tr.numpy())

    mu_tr = np.mean(mus_tr, axis=0)
    sd_tr = np.mean(sig_tr, axis=0)

    lo_tr, up_tr = mu_tr - sd_tr, mu_tr + sd_tr

    mu_tr *= scale
    lo_tr *= scale
    up_tr *= scale

    model.eval() 
    with torch.no_grad():
        mu_te, sd_te = model(Xs)
        
    lo_te, up_te = mu_te - sd_te, mu_te + sd_te

    mu_te *= scale
    lo_te *= scale
    up_te *= scale

    def build(X, Y, mu, lo, up):
        return [{
            "history": X[i],
            "y_true": Y[i],
            "y_pred": mu[i],
            "lower": lo[i],
            "upper": up[i],
        } for i in range(len(X))]

    res_tr = build(X_tr, Y_tr, mu_tr, lo_tr, up_tr)
    res_te = build(X_te, Y_te, mu_te, lo_te, up_te)

    rmse, cov = evaluate_forecasts(res_te)
    rmse_h_te, cov_h_te = evaluate_by_horizon(res_te)
    _, cov_h_tr = evaluate_by_horizon(res_tr)
    wink = compute_winkler(res_te)

    return {
        "results_train": res_tr,
        "results_test": res_te,
        "rmse": rmse,
        "coverage": cov,
        "winkler": wink,
        "coverage_h_train": cov_h_tr,
        "coverage_h_test": cov_h_te,
    }


In [ ]:
# RUN DeepAR on ALL datasets

data = np.load("time_series/synthetic_time_series.npz")

summary = []

for key in data.files:
    print(f"\nRunning DeepAR on {key}")
    out = run_deepar_pipeline(data[key])
    if out is None:
        continue

    plot_best_worst_forecasts_deepar(out["results_test"], key)
    plot_coverage_train_vs_test(
        out["coverage_h_train"],
        out["coverage_h_test"],
        key
    )

    summary.append({
        "dataset": key,
        "RMSE": out["rmse"],
        "Coverage": out["coverage"],
        "Winkler": out["winkler"],
    })

df_deepar = pd.DataFrame(summary)
print("\n✅ DeepAR Results Summary")
print(df_deepar)